# Перцентили зарплат Avito на приоритетных сегментах

[Контракт ML](../Analytics/salary_segment_results/ml_salary_contract.json) ·
[Уточнение приоритетов от 16 ноября](../Analytics/salary_segment_normality.ipynb) ·
[Строгий пересчёт нагрузки](../Analytics/salary_segment_results/priority_salary_formats_and_collection.md) ·
[Сценарий 176 часов](../Analytics/salary_normalization_v1.ipynb)

Кассиры Санкт-Петербурга проверяются первыми по постановке задачи. В сводке аналитиков
продавцы и комплектовщики Санкт-Петербурга готовы по качеству данных. Московские
комплектовщики с 11–12 часами остаются кандидатом на добор. Проверка ниже
использует пять профессий × три согласованных города. Вахта, удалёнка и военные
вакансии исключены. Положительные точные суммы и середины закрытых вилок служат
точками. Период и налоговая база должны быть явно указаны.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next(p for p in (cwd, *cwd.parents) if (p / 'ML/run_salary_gold.py').exists())
folder = root / 'ML/experiments/2026-11-18_priority_salary_gold'
rerun = False
if rerun:
    subprocess.run([sys.executable, 'ML/run_salary_gold.py'], cwd=root, check=True)
    subprocess.run([sys.executable, 'ML/run_salary_gold_v1.py'], cwd=root, check=True)

def read(name):
    return pd.read_csv(folder / (name + '.csv'), dtype={'id': str})

passport = json.loads((folder / 'experiment.json').read_text(encoding='utf-8'))
metrics = read('metrics')
gold = read('gold')
cashier = 'Кассир × Санкт-Петербург'
seller = 'Продавец × Санкт-Петербург'
picker = 'Комплектовщик × Санкт-Петербург'
key_scopes = [cashier, seller, picker, 'gold', 'ordinary']
display(gold.groupby(['unit', 'tax']).agg(точки=('id', 'size'), группы=('group', 'nunique')))
print(f"Каноническая выгрузка: {passport['dataset']['rows']} вакансий")


,,точки,группы
unit,tax,,
hour,net,13,13
shift,net,785,396
week,net,11,11


Каноническая выгрузка: 32097 вакансий


## Цель, разбиение и статистический ориентир

Каждая вакансия получает один прогноз из трёх отложенных групп работодателей.
Группы объединяют известные профили и имена, а также точные совпадения город /
профессия / заголовок / описание. На каждой итерации из остальных групп
выделена валидация. Параметры карточки и очищенный текст доступны модели.
Зарплатные поля, работодатели, фильтры поиска и просмотры в признаки не входят.

Эмпирические Q25/P50/P75 считаются из обучения. Когда в ячейке менее пяти
точек или трёх групп, берём профессию, затем все сопоставимые строки той же
единицы и базы. Показатели аналитиков из полного текущего среза приводятся
для описания данных. Средний pinball — средняя ошибка P25/P50/P75 в одной
единице оплаты. Метрики по всему gold и обычному holdout рассчитываются
одинаково. Для интервала P25–P75 ориентир покрытия — 50%.

In [2]:
summary = read('support')
fields = ['scope', 'n', 'parsed', 'employers', 'q1', 'median', 'q3', 'class_a',
          'to_30', 'to_50', 'strict_shift_points', 'strict_to_50_by_volume']
display(summary.loc[summary.scope.isin([*key_scopes[:3], 'cashier_spb_permanent']), fields].round(1))

shown = ['scope', 'model', 'n', 'groups', 'pinball', 'mae', 'coverage', 'width']
main = metrics.query("branch == 'main'")
display(main.loc[main.scope.eq(cashier), shown].sort_values('pinball').round(2))
variants = ['empirical', 'chewby', 'ssr_text', 'cat_detail', 'cat_relative', 'cat_local']
display(main.loc[main.scope.isin(key_scopes) & main.model.isin(variants)]
        .pivot(index='model', columns='scope', values='pinball').reindex(index=variants, columns=key_scopes).round())


,scope,n,parsed,employers,q1,median,q3,class_a,to_30,to_50,strict_shift_points,strict_to_50_by_volume
1,Кассир × Санкт-Петербург,34,129,30,3250.0,3903.0,4750.0,False,61,61,9,41
4,Продавец × Санкт-Петербург,107,208,66,3687.5,4000.0,5025.0,True,0,0,19,31
10,Комплектовщик × Санкт-Петербург,97,187,52,4100.0,4625.0,5000.0,True,0,0,9,41
15,cashier_spb_permanent,27,74,23,3425.0,4000.0,4762.5,False,20,23,8,42


,scope,model,n,groups,pinball,mae,coverage,width
88,Кассир × Санкт-Петербург,cat_detail_uncalibrated,34,30,357.88,802.92,0.41,951.85
122,Кассир × Санкт-Петербург,cat_relative,34,30,368.06,831.60,0.56,1319.80
71,Кассир × Санкт-Петербург,cat_detail,34,30,371.19,823.11,0.50,1478.34
20,Кассир × Санкт-Петербург,chewby,34,30,386.42,857.06,0.53,1237.30
3,Кассир × Санкт-Петербург,empirical,34,30,396.89,892.21,0.53,1505.00
105,Кассир × Санкт-Петербург,ssr_text,34,30,399.35,880.04,0.56,1641.35
54,Кассир × Санкт-Петербург,cat_core,34,30,401.00,924.49,0.47,1252.10
37,Кассир × Санкт-Петербург,cat_matched,34,30,438.03,975.64,0.50,1499.47
137,Кассир × Санкт-Петербург,cat_local,34,30,524.66,1142.39,0.09,636.20


scope,Кассир × Санкт-Петербург,Продавец × Санкт-Петербург,Комплектовщик × Санкт-Петербург,gold,ordinary
model,,,,,
empirical,397.0,2206.0,902.0,1518.0,740.0
chewby,386.0,2198.0,905.0,1557.0,736.0
ssr_text,399.0,2170.0,900.0,1536.0,735.0
cat_detail,371.0,2215.0,888.0,1552.0,741.0
cat_relative,368.0,2192.0,879.0,1530.0,723.0
cat_local,525.0,2224.0,908.0,1445.0,NaN


## Парные проверки, признаки и ошибки

Разность pinball считается на одних ID. Бутстреп пересэмплирует группы
работодателей поверх сохранённых прогнозов. Доверительные интервалы
описывают сравнения в этом эксперименте и не учитывают поиск нескольких
вариантов. Общий итог локальных моделей содержит только три ячейки;
сравнивать его следует с теми же строками общей модели в парной таблице.

Перестановка одного поля не заменяет эксперимент с удалением целой группы
взаимосвязанных признаков. График, занятость и другие поля переставляются
внутри профессии и города. Профессия и город переставляются по всему gold.
У самых больших ошибок проверяйте единицу и контекст карточки.

In [3]:
pair = read('paired_comparisons')
display(pair.query("branch == 'main'").loc[lambda t: t.scope.isin(['gold', cashier, seller])].round(2))
features = read('feature_permutation')
display(features.groupby('field').delta_pinball.agg(['mean','min','max']).sort_values('mean', ascending=False).round(2))
display(read('subgroups').query("model in ['empirical','cat_detail','ssr_text']").round(2))
display(read('salary_form_errors').query("model in ['empirical','chewby','ssr_text','cat_detail']").round(2))
errors = read('largest_errors')
display(errors.query("model == 'empirical'").nlargest(8, 'absolute_error_p50')
        [['id','profession_resolved','city_resolved','shape','actual','p50','absolute_error_p50']])


,branch,scope,alternative,reference,n,groups,delta,ci_low,ci_high,change_pct
1,main,gold,chewby,empirical,785,396,38.58,-11.00,102.46,2.54
2,main,Кассир × Санкт-Петербург,chewby,empirical,34,30,-10.47,-35.27,15.69,-2.64
3,main,Продавец × Санкт-Петербург,chewby,empirical,107,67,-7.28,-20.22,6.87,-0.33
6,main,gold,cat_matched,chewby,785,396,16.17,-21.13,50.68,1.04
7,main,Кассир × Санкт-Петербург,cat_matched,chewby,34,30,51.61,-15.02,117.41,13.36
8,main,Продавец × Санкт-Петербург,cat_matched,chewby,107,67,56.75,-77.95,152.98,2.58
11,main,gold,cat_detail,cat_core,785,396,-33.30,-60.62,-5.65,-2.10
12,main,Кассир × Санкт-Петербург,cat_detail,cat_core,34,30,-29.81,-103.98,43.34,-7.43
13,main,Продавец × Санкт-Петербург,cat_detail,cat_core,107,67,9.93,-36.72,50.47,0.45
16,main,gold,ssr_text,cat_detail,785,396,-15.64,-34.01,4.64,-1.01


,mean,min,max
field,,,
profession_resolved,136.08,85.15,218.91
city_resolved,26.12,10.86,53.29
Количество рабочих часов в день,8.46,0.51,16.32
Способ оформления,3.24,-3.89,7.59
Занятость,1.90,-1.87,4.28
Медкнижка,1.80,-1.28,4.46
График,1.04,-5.21,7.59
Опыт работы,-3.05,-4.56,-0.53


,subgroup,part,model,n,available_n,groups,pinball,pinball_p25,pinball_p50,pinball_p75,mae,medae,coverage,width,below_p25,below_p50,below_p75
1,cashier_spb_permanent,selected,cat_detail,27,27,23,370.83,299.79,413.13,399.57,826.26,572.77,0.52,1478.25,0.37,0.48,0.89
7,cashier_spb_permanent,selected,empirical,27,27,23,430.92,349.04,486.33,457.38,972.67,750.00,0.56,1505.00,0.19,0.37,0.74
8,cashier_spb_permanent,selected,ssr_text,27,27,23,418.27,319.11,464.37,471.33,928.75,632.72,0.59,1707.96,0.30,0.63,0.89
10,cashier_spb_permanent,remainder,cat_detail,7,7,7,372.58,316.11,405.49,396.15,810.99,849.82,0.43,1481.17,0.43,0.57,0.86
16,cashier_spb_permanent,remainder,empirical,7,7,7,265.64,173.39,290.93,332.59,581.86,663.00,0.43,1505.00,0.43,0.71,0.86
17,cashier_spb_permanent,remainder,ssr_text,7,7,7,326.37,310.52,346.10,322.49,692.19,763.65,0.43,1574.73,0.43,0.71,0.86
19,cashier_spb_medical_help,selected,cat_detail,22,22,20,403.90,342.83,478.60,390.26,957.21,631.07,0.55,1509.39,0.32,0.41,0.86
25,cashier_spb_medical_help,selected,empirical,22,22,20,478.67,378.99,549.52,507.49,1099.05,850.00,0.45,1505.00,0.23,0.36,0.68
26,cashier_spb_medical_help,selected,ssr_text,22,22,20,464.43,374.14,524.61,494.54,1049.22,759.66,0.55,1780.10,0.32,0.55,0.86
28,cashier_spb_medical_help,remainder,cat_detail,12,12,11,311.23,230.40,288.64,414.65,577.28,405.25,0.42,1422.80,0.50,0.67,0.92


,model,salary_form,n,available_n,groups,pinball,pinball_p25,pinball_p50,pinball_p75,mae,medae,coverage,width,below_p25,below_p50,below_p75
2,cat_detail,exact,167,167,95,407.07,302.73,428.51,489.97,857.02,635.34,0.42,1354.49,0.50,0.80,0.92
3,cat_detail,range,618,618,334,1861.03,1083.66,1931.40,2568.02,3862.80,921.99,0.43,1583.75,0.34,0.49,0.77
12,chewby,exact,167,167,95,380.59,297.28,422.75,421.74,845.50,534.96,0.44,1238.34,0.38,0.69,0.83
13,chewby,range,618,618,334,1874.98,1145.30,1945.41,2534.23,3890.83,879.49,0.46,1596.52,0.28,0.46,0.74
14,empirical,exact,167,167,95,377.81,260.53,457.40,415.51,914.79,550.00,0.46,1315.00,0.45,0.78,0.87
15,empirical,range,618,618,334,1826.73,1041.93,1903.25,2535.00,3806.50,744.00,0.52,1315.00,0.22,0.48,0.72
16,ssr_text,exact,167,167,95,368.19,289.40,374.14,441.03,748.27,481.01,0.56,1496.20,0.38,0.68,0.94
17,ssr_text,range,618,618,334,1851.67,1069.84,1921.89,2563.28,3843.78,863.27,0.46,1741.90,0.33,0.51,0.80


,id,profession_resolved,city_resolved,shape,actual,p50,absolute_error_p50
6,7231222956,Водитель грузового транспорта,Санкт-Петербург,range,230000.0,12000.0,218000.0
10,2309383081,Водитель грузового транспорта,Санкт-Петербург,range,215000.0,8500.0,206500.0
16,8490218390,Продавец,Санкт-Петербург,range,135500.0,4000.0,131500.0
31,7580450201,Водитель грузового транспорта,Санкт-Петербург,range,135000.0,12000.0,123000.0
39,8254553567,Продавец,Москва,range,120000.0,4505.0,115495.0
41,8448362850,Комплектовщик,Санкт-Петербург,range,112625.0,4257.0,108368.0
50,8324344574,Продавец,Санкт-Петербург,range,106500.0,3850.0,102650.0
60,8082345866,Продавец,Екатеринбург,range,100000.0,3705.0,96295.0


## Месячные эквиваленты, период и относительная зарплата

Строгий пересчёт сохраняет оплату на руки. Для кассиров сравниваются ровно
те же сменные объявления с единственным документированным коэффициентом.
Обучение на сменных ставках с пересчётом прогнозов и обучение на месячной
цели оцениваются в месячных рублях. Третья ветка добавляет другие строго
пересчитываемые форматы только в обучение. Второй сценарий использует V1
аналитика, 176 часов и gross. Указанные закрытые интервалы часов заменяются
серединой. Импутация часов, несколько значений часов, неизвестный период и
односторонние границы исключены. Сценарные gross и строгие net не сравниваются между собой.

Относительный CatBoost учится на отношении зарплаты к медиане. Для строки
обучения медиана рассчитана без её группы работодателя. На проверке
предсказанное отношение умножается на медиану обучения. В отдельной
временной проверке веса заморожены. Данные до 29 октября служат обучению,
данные 30 октября — 13 ноября дают свежую медиану, более поздние
объявления остаются тестом. Работодатели более позднего теста удалены из
обучения и обновления.

In [4]:
for branch in ('normalization', 'pay_type', 'temporal'):
    display(metrics.loc[metrics.branch.eq(branch) & metrics.scope.isin(key_scopes),
                        ['metric_unit','metric_tax',*shown]].round(2))
display(read('v1_metrics').loc[lambda t: t.scope.isin([cashier, 'gold']), ['metric_unit','metric_tax',*shown]].round(2))
display(read('pay_type_paired').query("scope == 'gold'").round(2))
display(read('median_update_support').loc[lambda t: t.future.gt(0),
        ['scope','old_raw','update','future','future_with_fresh_anchor','median_old','median_updated']])


,metric_unit,metric_tax,scope,model,n,groups,pinball,mae,coverage,width
140,month,net,ordinary,native_to_month,70,67,15565.61,33070.54,0.51,23301.75
141,month,net,gold,native_to_month,105,93,29330.86,59893.21,0.52,25760.89
143,month,net,Кассир × Санкт-Петербург,native_to_month,9,9,6911.34,15011.50,0.67,27390.80
146,month,net,Продавец × Санкт-Петербург,native_to_month,19,19,47168.74,94858.04,0.58,27807.98
152,month,net,Комплектовщик × Санкт-Петербург,native_to_month,9,9,6528.37,14280.11,0.67,28190.87
155,month,net,ordinary,strict_month_target,70,67,16316.08,33954.69,0.51,25325.35
156,month,net,gold,strict_month_target,105,93,29615.11,60062.44,0.53,25790.90
158,month,net,Кассир × Санкт-Петербург,strict_month_target,9,9,6983.76,13843.98,0.44,23744.72
161,month,net,Продавец × Санкт-Петербург,strict_month_target,19,19,48481.64,97344.32,0.42,29550.84
167,month,net,Комплектовщик × Санкт-Петербург,strict_month_target,9,9,5968.60,12173.81,0.67,31122.94


,metric_unit,metric_tax,scope,model,n,groups,pinball,mae,coverage,width
185,shift,net,ordinary,pooled_month_with_unit,70,67,901.83,1923.20,0.53,1440.79
186,shift,net,gold,pooled_month_with_unit,105,93,1889.44,3884.42,0.50,1627.47
188,shift,net,Кассир × Санкт-Петербург,pooled_month_with_unit,9,9,533.15,1221.03,0.44,1627.47
191,shift,net,Продавец × Санкт-Петербург,pooled_month_with_unit,19,19,3162.36,6397.99,0.37,1489.88
197,shift,net,Комплектовщик × Санкт-Петербург,pooled_month_with_unit,9,9,352.48,908.31,0.56,1748.33
200,week,net,ordinary,pooled_month_with_unit,16,10,13859.53,28391.65,0.25,11994.74
201,week,net,gold,pooled_month_with_unit,11,11,21938.54,45535.71,0.45,13350.77
206,week,net,Комплектовщик × Санкт-Петербург,pooled_month_with_unit,3,3,2110.06,5535.05,1.00,13350.77
209,hour,net,ordinary,unit_empirical,26,17,145.56,338.85,0.35,465.00
210,hour,net,gold,unit_empirical,13,13,57.53,132.35,0.38,187.50


,metric_unit,metric_tax,scope,model,n,groups,pinball,mae,coverage,width
249,shift,net,gold,frozen_anchor,3,3,219.31,473.62,0.33,614.86
251,shift,net,gold,updated_anchor,3,3,219.31,473.62,0.33,614.86


,metric_unit,metric_tax,scope,model,n,groups,pinball,mae,coverage,width
1,month,gross,gold,native_to_v1_gross,382,230,20960.92,43394.51,0.48,18277.39
3,month,gross,Кассир × Санкт-Петербург,native_to_v1_gross,19,18,4124.10,9653.18,0.42,16032.08
18,month,gross,gold,v1_gross_target,382,230,21850.08,45079.61,0.40,16854.20
20,month,gross,Кассир × Санкт-Петербург,v1_gross_target,19,18,4923.66,11291.23,0.21,12059.88


,unit,branch,scope,alternative,reference,n,groups,delta,ci_low,ci_high,change_pct
0,hour,pay_type,gold,unit_cat,unit_empirical,13,13,16.41,-11.94,47.28,28.53
1,hour,pay_type,gold,pooled_month_with_unit,unit_empirical,1,1,-157.38,NaN,NaN,-66.27
2,hour,pay_type,gold,pooled_month_with_unit,unit_cat,1,1,-258.10,NaN,NaN,-76.31
4,week,pay_type,gold,unit_cat,unit_empirical,11,11,-660.26,-2688.89,1544.78,-2.79
5,week,pay_type,gold,pooled_month_with_unit,unit_empirical,11,11,-1761.70,-4181.41,885.24,-7.43
6,week,pay_type,gold,pooled_month_with_unit,unit_cat,11,11,-1101.44,-4103.18,1834.53,-4.78


,scope,old_raw,update,future,future_with_fresh_anchor,median_old,median_updated
0,gold,656,50,50,3,NaN,NaN
1,Кассир × Москва,22,0,9,0,4125.0,NaN
2,Кассир × Санкт-Петербург,29,0,5,0,3750.0,NaN
3,Кассир × Екатеринбург,40,26,3,3,2310.0,2310.0
4,Продавец × Москва,53,1,17,0,4505.0,NaN
6,Продавец × Екатеринбург,20,1,7,0,2155.0,NaN
7,Грузчик × Москва,52,7,6,0,4350.0,NaN
10,Комплектовщик × Москва,87,0,1,0,5500.0,NaN
14,Водитель грузового транспорта × Санкт-Петербург,25,0,2,0,10000.0,NaN


## Артефакты и решение

[Сводка и дерево вариантов](experiments/2026-11-18_priority_salary_gold/README.md) ·
[Золотая выборка](experiments/2026-11-18_priority_salary_gold/gold.csv) ·
[Метрики gold](experiments/2026-11-18_priority_salary_gold/metrics_gold.csv) ·
[Инструкция и пакеты](models/2026-11-18_priority_salary_gold/README.md)

На всём золотом сменном срезе наименьшая средняя ошибка у эмпирических квартилей.
Проверка кассиров Санкт-Петербурга не даёт основания назначать им отдельную модель.
Для продавцов Санкт-Петербурга CatBoost с текстом имеет небольшой выигрыш
и заслуживает проверки на новом независимом срезе.
Нормализация и относительный таргет остаются исследовательскими ветками.
